<a href="https://colab.research.google.com/github/maivu18082005-hub/Parameter_Identification_of_PMSM_using_PINN/blob/main/05_hybrid_pinn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ===== CELL 1: KẾT NỐI DRIVE & KIỂM TRA GPU =====
from google.colab import drive  # Import thư viện drive của Colab để gắn kết nối tới Google Drive
import torch                    # Import thư viện PyTorch làm framework chính cho Deep Learning
import os                       # Import thư viện os để thực hiện các thao tác kiểm tra thư mục và đường dẫn

drive.mount('/content/drive')   # Thực hiện Mount (gắn) Google Drive của bạn vào môi trường ảo Colab

# Khai báo đường dẫn thư mục dự án gốc trên Google Drive
DRIVE_PROJECT = '/content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN'

# Khai báo đường dẫn chứa tập dữ liệu đã qua tiền xử lý (.npy và .pkl)
PROCESSED_DATA_DIR = os.path.join(DRIVE_PROJECT, 'dataset', 'processed')

# Khai báo thư mục lưu trữ trọng số mô hình tốt nhất (Checkpoints)
CHECKPOINT_DIR = os.path.join(DRIVE_PROJECT, 'checkpoints')

# Tự động tạo thư mục checkpoints trên Google Drive nếu thư mục này chưa tồn tại
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Kiểm tra xem Colab có GPU không: nếu có thì dùng 'cuda', nếu không thì dùng 'cpu'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Thiết bị tính toán đang dùng:', device) # In ra màn hình thiết bị đang chạy (ví dụ: cuda:0)

Mounted at /content/drive
Thiết bị tính toán đang dùng: cpu


In [14]:
# ===== BƯỚC 1: MẠNG PINN CHUYÊN BIỆT NHẬN DẠNG [Ld, Lq] =====
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import joblib
import time

# 1. Nạp dữ liệu 5 đặc trưng chuẩn lên GPU
X_train_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'X_train.npy')), dtype=torch.float32, device=device)
Y_train_2d = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'Y_train.npy'))[:, 1:3], dtype=torch.float32, device=device) # Chỉ lấy cột Ld, Lq
S_train_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'Steady_train.npy')), dtype=torch.float32, device=device)

X_val_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'X_validation.npy')), dtype=torch.float32, device=device)
Y_val_2d = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'Y_validation.npy'))[:, 1:3], dtype=torch.float32, device=device)

X_test_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'X_test.npy')), dtype=torch.float32, device=device)
Y_test_full = np.load(os.path.join(PROCESSED_DATA_DIR, 'Y_test.npy'))

X_scaler = joblib.load(os.path.join(PROCESSED_DATA_DIR, 'X_scaler.pkl'))
Y_scaler = joblib.load(os.path.join(PROCESSED_DATA_DIR, 'Y_scaler.pkl'))

x_mean = torch.tensor(X_scaler.mean_, dtype=torch.float32, device=device)
x_scale = torch.tensor(X_scaler.scale_, dtype=torch.float32, device=device)
ldq_mean = torch.tensor(Y_scaler.mean_[1:3], dtype=torch.float32, device=device)
ldq_scale = torch.tensor(Y_scaler.scale_[1:3], dtype=torch.float32, device=device)

# 2. Kiến trúc mạng chuyên biệt 2 đầu ra [Ld, Lq]
class PMSM_Inductance_Net(nn.Module):
    def __init__(self, input_dim=5, output_dim=2, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x):
        return self.net(x)

model_ldq = PMSM_Inductance_Net().to(device)
optimizer = optim.Adam(model_ldq.parameters(), lr=0.002)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=15)
criterion = nn.MSELoss()

# 3. Huấn luyện 200 epochs trên GPU
epochs = 200
batch_size = 512
n_train = len(X_train_t)
best_val = float('inf')

start_time = time.time()
for epoch in range(1, epochs + 1):
    model_ldq.train()
    perm = torch.randperm(n_train, device=device)
    for i in range(0, n_train, batch_size):
        idx = perm[i:i+batch_size]
        pred = model_ldq(X_train_t[idx])
        loss = criterion(pred, Y_train_2d[idx])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model_ldq.eval()
    with torch.no_grad():
        val_loss = criterion(model_ldq(X_val_t), Y_val_2d).item()
    scheduler.step(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model_ldq.state_dict(), '/content/best_ldq_model.pth')

    if epoch % 40 == 0 or epoch == 1:
        print(f"Epoch [{epoch:3d}/{epochs}] | Val Loss (Ld, Lq): {val_loss:.6f} | Thời gian: {time.time()-start_time:.1f}s")

Epoch [  1/200] | Val Loss (Ld, Lq): 0.001245 | Thời gian: 0.7s
Epoch [ 40/200] | Val Loss (Ld, Lq): 0.000118 | Thời gian: 31.6s
Epoch [ 80/200] | Val Loss (Ld, Lq): 0.000098 | Thời gian: 63.0s
Epoch [120/200] | Val Loss (Ld, Lq): 0.000064 | Thời gian: 93.5s
Epoch [160/200] | Val Loss (Ld, Lq): 0.000035 | Thời gian: 123.8s
Epoch [200/200] | Val Loss (Ld, Lq): 0.000029 | Thời gian: 154.8s


In [15]:
# ===== CELL 3 (ĐÃ SỬA): CHỈ GIẢI Rs TẠI CÁC ĐIỂM XÁC LẬP CHUẨN (STEADY == 1) =====
import pandas as pd

# 1. Nạp model tốt nhất và dự đoán Ld, Lq trên tập Test
model_ldq.load_state_dict(torch.load('/content/best_ldq_model.pth'))
model_ldq.eval()

with torch.no_grad():
    pred_ldq_scaled = model_ldq(X_test_t).cpu().numpy()

Ld_pred = pred_ldq_scaled[:, 0] * Y_scaler.scale_[1] + Y_scaler.mean_[1]
Lq_pred = pred_ldq_scaled[:, 1] * Y_scaler.scale_[2] + Y_scaler.mean_[2]

# Giải chuẩn hóa Input Test & Target thực tế
X_test_real = X_test_t.cpu().numpy() * X_scaler.scale_ + X_scaler.mean_
ud, uq, Id, Iq, nm = X_test_real[:, 0], X_test_real[:, 1], X_test_real[:, 2], X_test_real[:, 3], X_test_real[:, 4]

Y_test_real = Y_scaler.inverse_transform(Y_test_full)
Rs_true, Ld_true, Lq_true = Y_test_real[:, 0], Y_test_real[:, 1], Y_test_real[:, 2]

# Nạp cờ xác lập chuẩn của tập Test (đã lọc bỏ quá độ dI/dt != 0 và S_FW != 0)
steady_test = np.load(os.path.join(PROCESSED_DATA_DIR, 'Steady_test.npy')) > 0.5

# 2. Tính sụt áp thuần trở theo đúng thông số động cơ (p = 9, psi_m = 0.2319 Wb)
pole_pairs = 9
psi_m = 0.2319
omega_e = pole_pairs * nm * (2.0 * np.pi / 60.0)

delta_ud = ud + omega_e * Lq_pred * Iq
delta_uq = uq - omega_e * (Ld_pred * Id + psi_m)

# 3. CHỈ ước lượng Rs tại các điểm xác lập có dòng tải lớn (|Iq| > 15 A) để tránh nhiễu PWM và quá độ L*di/dt
Rs_raw = np.full_like(Rs_true, np.nan)
high_load_steady = steady_test & (np.abs(Iq) > 15.0)

# Tính Rs tức thời tại các điểm xác lập tải lớn và làm mượt bằng trung bình trượt
Rs_raw[high_load_steady] = (Id[high_load_steady] * delta_ud[high_load_steady] + Iq[high_load_steady] * delta_uq[high_load_steady]) / (Id[high_load_steady]**2 + Iq[high_load_steady]**2)

# Chặn nhiễu ngoại lai trong dải vật lý hợp lý của Rs [0.030, 0.050] Ohm
Rs_raw = np.where((Rs_raw >= 0.030) & (Rs_raw <= 0.050), Rs_raw, np.nan)

# Làm mượt chuỗi Rs hợp lệ và giữ giá trị gần nhất (Sample-and-Hold) cho vùng không tải / quá độ
Rs_phys = pd.Series(Rs_raw).rolling(50, min_periods=1).mean().ffill().bfill().to_numpy()

# 4. Đánh giá sai số MAPE
mape_Rs = np.mean(np.abs((Rs_true - Rs_phys) / Rs_true)) * 100
mape_Ld = np.mean(np.abs((Ld_true - Ld_pred) / Ld_true)) * 100
mape_Lq = np.mean(np.abs((Lq_true - Lq_pred) / Lq_true)) * 100

print("=== KẾT QUẢ MÔ HÌNH LAI (SAU KHI LỌC ĐÚNG VÙNG XÁC LẬP) ===")
print(f"Sai số Rs trung bình (Giải vật lý): {mape_Rs:.2f}%")
print(f"Sai số Ld trung bình (Mạng Neural): {mape_Ld:.2f}%")
print(f"Sai số Lq trung bình (Mạng Neural): {mape_Lq:.2f}%")

=== KẾT QUẢ MÔ HÌNH LAI (SAU KHI LỌC ĐÚNG VÙNG XÁC LẬP) ===
Sai số Rs trung bình (Giải vật lý): 7.78%
Sai số Ld trung bình (Mạng Neural): 0.07%
Sai số Lq trung bình (Mạng Neural): 0.09%


In [17]:
# ===== THỬ NGHIỆM: MẠNG PINN CHUYÊN BIỆT 1 ĐẦU RA CHO Rs (TỐI ƯU TRỰC TIẾP MAPE) =====
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import joblib
import time

# Nạp dữ liệu 5 đặc trưng chuẩn
X_train_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'X_train.npy')), dtype=torch.float32, device=device)
Y_train_rs = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'Y_train.npy'))[:, 0:1], dtype=torch.float32, device=device)
S_train_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'Steady_train.npy')), dtype=torch.float32, device=device)

X_val_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'X_validation.npy')), dtype=torch.float32, device=device)
Y_val_rs = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'Y_validation.npy'))[:, 0:1], dtype=torch.float32, device=device)

X_test_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'X_test.npy')), dtype=torch.float32, device=device)
Y_test_rs = np.load(os.path.join(PROCESSED_DATA_DIR, 'Y_test.npy'))[:, 0]

X_scaler = joblib.load(os.path.join(PROCESSED_DATA_DIR, 'X_scaler.pkl'))
Y_scaler = joblib.load(os.path.join(PROCESSED_DATA_DIR, 'Y_scaler.pkl'))

x_mean = torch.tensor(X_scaler.mean_, dtype=torch.float32, device=device)
x_scale = torch.tensor(X_scaler.scale_, dtype=torch.float32, device=device)
rs_mean = float(Y_scaler.mean_[0])
rs_scale = float(Y_scaler.scale_[0])

class Rs_PINN(nn.Module):
    def __init__(self, input_dim=5, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.SiLU(),
            nn.Dropout(0.1),  # Chống học vẹt tập Train Data1-3
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, x):
        return self.net(x)

def rs_physics_loss(inputs_scaled, rs_pred_scaled, steady_mask, pole_pairs=9, psi_m=0.2319):
    mask = steady_mask > 0.5
    if mask.sum() == 0:
        return torch.tensor(0.0, device=inputs_scaled.device)
    in_r = inputs_scaled[mask] * x_scale + x_mean
    ud, uq, Id, Iq, nm = in_r[:, 0], in_r[:, 1], in_r[:, 2], in_r[:, 3], in_r[:, 4]
    Rs_est = rs_pred_scaled[mask, 0] * rs_scale + rs_mean

    # Dùng giá trị điện cảm danh định trung bình Ld ~ 0.00115 H, Lq ~ 0.00138 H
    omega_e = pole_pairs * nm * (2.0 * np.pi / 60.0)
    uq_calc = Rs_est * Iq + omega_e * (0.00115 * Id + psi_m)
    return torch.mean(((uq - uq_calc) / 10.0) ** 2)

# Hàm mất mát tối ưu trực tiếp sai số phần trăm thực tế của Rs
def rs_mape_loss(pred_scaled, target_scaled):
    pred_real = pred_scaled * rs_scale + rs_mean
    target_real = target_scaled * rs_scale + rs_mean
    return torch.mean(torch.abs((target_real - pred_real) / target_real)) * 100.0

model_rs = Rs_PINN().to(device)
optimizer = optim.AdamW(model_rs.parameters(), lr=0.001, weight_decay=1e-2)
epochs = 100
batch_size = 512
n_train = len(X_train_t)
best_val_mape = float('inf')

for epoch in range(1, epochs + 1):
    model_rs.train()
    perm = torch.randperm(n_train, device=device)
    for i in range(0, n_train, batch_size):
        idx = perm[i:i+batch_size]
        pred = model_rs(X_train_t[idx])
        loss_d = rs_mape_loss(pred, Y_train_rs[idx])
        loss_p = rs_physics_loss(X_train_t[idx], pred, S_train_t[idx])
        loss = loss_d + 0.5 * loss_p
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model_rs.eval()
    with torch.no_grad():
        val_mape = rs_mape_loss(model_rs(X_val_t), Y_val_rs).item()
    if val_mape < best_val_mape:
        best_val_mape = val_mape
        torch.save(model_rs.state_dict(), '/content/best_rs_model.pth')

model_rs.load_state_dict(torch.load('/content/best_rs_model.pth'))
model_rs.eval()
with torch.no_grad():
    pred_rs_real = model_rs(X_test_t).cpu().numpy()[:, 0] * rs_scale + rs_mean

true_rs_real = Y_test_rs * rs_scale + rs_mean
test_mape_rs = np.mean(np.abs((true_rs_real - pred_rs_real) / true_rs_real)) * 100
print(f"Sai số Rs trung bình trên tập Test (Tối ưu trực tiếp MAPE + Dropout): {test_mape_rs:.2f}%")

Sai số Rs trung bình trên tập Test (Tối ưu trực tiếp MAPE + Dropout): 5.48%
